# Feature Importance & SHAP Analysis

Analyze which features contribute most to each outcome model (K, BB, HR, 1B, 2B, 3B, OUT).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from pathlib import Path
from src.model.train_binary_models import BinaryModelEnsemble, OUTCOME_CLASSES

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')

print(f"Outcome classes: {OUTCOME_CLASSES}")

In [ ]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

## 1. Load Models

In [ ]:
model_dir = Path("../models/binary_ensemble")

ensemble = BinaryModelEnsemble.load(model_dir)

print(f"Loaded ensemble with {len(ensemble.models)} models")
print(f"\nFeatures per model:")
for outcome in OUTCOME_CLASSES:
    n_features = len(ensemble.selected_features.get(outcome, []))
    print(f"  {outcome}: {n_features} features")

## 2. Feature Importance (Built-in)

LightGBM's built-in feature importance based on split gain.

In [ ]:
# Get feature importance for all models
importance_dfs = {}

for outcome in OUTCOME_CLASSES:
    df = ensemble.get_feature_importance_df(outcome, save_dir=model_dir)
    if not df.empty:
        importance_dfs[outcome] = df
        print(f"\n{outcome} - Top 10 features:")
        print(df.head(10).to_string(index=False))

In [ ]:
# Plot feature importance for each outcome
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()

for i, outcome in enumerate(OUTCOME_CLASSES):
    ax = axes[i]
    
    if outcome in importance_dfs:
        df = importance_dfs[outcome].head(15)
        ax.barh(df['feature'], df['importance'], color=plt.cm.viridis(i/len(OUTCOME_CLASSES)))
        ax.set_xlabel('Importance')
        ax.set_title(f'{outcome} Model - Top 15 Features')
        ax.invert_yaxis()
    else:
        ax.text(0.5, 0.5, f'No data for {outcome}', ha='center', va='center')
        ax.set_title(f'{outcome} Model')

# Hide extra subplots
for j in range(len(OUTCOME_CLASSES), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/feature_importance_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Compare Features Across Models

Which features are important across multiple outcomes?

In [ ]:
# Combine all importances
all_importance = []

for outcome, df in importance_dfs.items():
    df = df.copy()
    df['outcome'] = outcome
    # Normalize importance within each model
    df['importance_norm'] = df['importance'] / df['importance'].max()
    all_importance.append(df)

combined = pd.concat(all_importance, ignore_index=True)

# Pivot to see feature importance across outcomes
pivot = combined.pivot_table(
    index='feature', 
    columns='outcome', 
    values='importance_norm',
    aggfunc='first'
).fillna(0)

# Calculate mean importance across models
pivot['mean_importance'] = pivot.mean(axis=1)
pivot['n_models'] = (pivot[OUTCOME_CLASSES] > 0).sum(axis=1)

# Top features used across multiple models
print("Features important across multiple models:")
multi_model = pivot[pivot['n_models'] >= 3].sort_values('mean_importance', ascending=False)
print(multi_model[['mean_importance', 'n_models']].head(20).to_string())

In [ ]:
# Heatmap of top features across models
top_features = pivot.sort_values('mean_importance', ascending=False).head(25).index

plt.figure(figsize=(12, 10))
sns.heatmap(
    pivot.loc[top_features, OUTCOME_CLASSES],
    cmap='YlOrRd',
    annot=True,
    fmt='.2f',
    cbar_kws={'label': 'Normalized Importance'}
)
plt.title('Feature Importance Across Outcome Models')
plt.xlabel('Outcome')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('../reports/feature_importance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Analysis

SHAP values show how each feature pushes the prediction higher or lower.

In [ ]:
# Load data and build enriched matchup features for SHAP
from src.data.preprocess import MatchupPreprocessor
from src.config import PITCHER_ROLLING_WINDOWS, BATTER_ROLLING_WINDOWS

# Load profiles
profiles_dir = Path('../data/profiles')
pitcher_profiles = pd.read_csv(profiles_dir / 'pitcher_arsenal.csv')
batter_profiles = pd.read_csv(profiles_dir / 'batter_profiles.csv')
print(f"Loaded {len(pitcher_profiles)} pitcher profiles, {len(batter_profiles)} batter profiles")

# Load raw pitches (has 'pitcher' and 'batter' columns needed by build_matchup_data)
pitches = pd.read_parquet('../data/raw/pitches.parquet')
print(f"Loaded {len(pitches):,} pitches")

# Sample for SHAP (full dataset is too slow/memory intensive)
# Get unique game_pks and sample from those to get complete plate appearances
sample_games = pitches['game_pk'].drop_duplicates().sample(n=50, random_state=42)
pitches_sample = pitches[pitches['game_pk'].isin(sample_games)].copy()
print(f"Sampled {len(pitches_sample):,} pitches from {len(sample_games)} games")

del pitches
clear_mem()

In [ ]:
# Build enriched matchup data (same as training pipeline)
preprocessor = MatchupPreprocessor()

print("Building matchup features (this may take a minute)...")
matchups = preprocessor.build_matchup_data(
    pitches_df=pitches_sample,
    pitcher_profiles=pitcher_profiles,
    batter_profiles=batter_profiles,
    pitcher_rolling_windows=PITCHER_ROLLING_WINDOWS,
    batter_rolling_windows=BATTER_ROLLING_WINDOWS,
)
print(f"Built {len(matchups):,} matchups with {len(matchups.columns)} columns")

# Filter to valid outcomes (same as training)
valid_outcomes = ['strikeout', 'walk', 'single', 'double', 'triple', 'home_run', 
                  'field_out', 'grounded_into_double_play', 'force_out', 'sac_fly', 'fielders_choice_out']
matchups = matchups[matchups['events'].isin(valid_outcomes)].copy()
print(f"Filtered to {len(matchups):,} valid matchups")

# Get feature columns (all features the models were trained on)
X_sample = matchups[ensemble.feature_names].copy()
print(f"X_sample shape: {X_sample.shape}")

# Cleanup
del pitches_sample, pitcher_profiles, batter_profiles, matchups, preprocessor
clear_mem()

In [ ]:
# SHAP for strikeout model (K)
outcome = 'K'
print(f"Computing SHAP values for {outcome} model...")

# Get model and features
model = ensemble.load_model(outcome, save_dir=model_dir)
features = ensemble.selected_features[outcome]
X_model = X_sample[features]

# Access underlying LightGBM model (FLAML wraps it)
lgb_model = model.model.estimator

# Compute SHAP
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer(X_model)

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_model, max_display=20, show=False)
plt.title(f'SHAP Summary - {outcome} (Strikeout) Model')
plt.tight_layout()
plt.savefig(f'../reports/shap_summary_{outcome}.png', dpi=150, bbox_inches='tight')
plt.show()

del model, lgb_model, explainer, shap_values
clear_mem()

In [ ]:
# SHAP for home run model (HR)
outcome = 'HR'
print(f"Computing SHAP values for {outcome} model...")

model = ensemble.load_model(outcome, save_dir=model_dir)
features = ensemble.selected_features[outcome]
X_model = X_sample[features]

lgb_model = model.model.estimator
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer(X_model)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_model, max_display=20, show=False)
plt.title(f'SHAP Summary - {outcome} (Home Run) Model')
plt.tight_layout()
plt.savefig(f'../reports/shap_summary_{outcome}.png', dpi=150, bbox_inches='tight')
plt.show()

del model, lgb_model, explainer, shap_values
clear_mem()

In [ ]:
# SHAP for walk model (BB)
outcome = 'BB'
print(f"Computing SHAP values for {outcome} model...")

model = ensemble.load_model(outcome, save_dir=model_dir)
features = ensemble.selected_features[outcome]
X_model = X_sample[features]

lgb_model = model.model.estimator
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer(X_model)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_model, max_display=20, show=False)
plt.title(f'SHAP Summary - {outcome} (Walk) Model')
plt.tight_layout()
plt.savefig(f'../reports/shap_summary_{outcome}.png', dpi=150, bbox_inches='tight')
plt.show()

del model, lgb_model, explainer, shap_values
clear_mem()

In [ ]:
# SHAP for remaining models (1B, 2B, 3B, OUT)
remaining = ['1B', '2B', '3B', 'OUT']

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, outcome in enumerate(remaining):
    print(f"Computing SHAP values for {outcome} model...")
    
    model = ensemble.load_model(outcome, save_dir=model_dir)
    features = ensemble.selected_features[outcome]
    X_model = X_sample[features]
    
    lgb_model = model.model.estimator
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer(X_model)
    
    plt.sca(axes[i])
    shap.summary_plot(shap_values, X_model, max_display=15, show=False)
    axes[i].set_title(f'{outcome} Model')
    
    del model, lgb_model, explainer, shap_values
    gc.collect()

plt.tight_layout()
plt.savefig('../reports/shap_summary_other.png', dpi=150, bbox_inches='tight')
plt.show()

clear_mem()

## 5. Summary Statistics

In [ ]:
# Summary table
summary_data = []

for outcome in OUTCOME_CLASSES:
    if outcome in importance_dfs:
        df = importance_dfs[outcome]
        top_feature = df.iloc[0]['feature']
        top_importance = df.iloc[0]['importance']
        n_features = len(df)
        
        summary_data.append({
            'Outcome': outcome,
            'N Features': n_features,
            'Top Feature': top_feature,
            'Top Importance': f"{top_importance:.3f}"
        })

summary_df = pd.DataFrame(summary_data)
print("\nModel Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# Save feature importance tables
reports_dir = Path('../reports')
reports_dir.mkdir(exist_ok=True)

# Save combined importance
combined.to_csv(reports_dir / 'feature_importance_all.csv', index=False)
print(f"Saved feature importance to {reports_dir / 'feature_importance_all.csv'}")

# Save pivot table
pivot.to_csv(reports_dir / 'feature_importance_pivot.csv')
print(f"Saved pivot table to {reports_dir / 'feature_importance_pivot.csv'}")

In [ ]:
# Final cleanup
del X_sample, ensemble, importance_dfs, combined, pivot
clear_mem()

print("\nAnalysis complete! Check ../reports/ for saved plots and CSVs.")